# Load input data

In [1]:
from functions import get_experiment_data

X_df4, meta_df4, batch_df4, _ = get_experiment_data(
    design_id="df4" ,
    file_path="/DATA/WGS_study/YSL/projects/Data",
    verbose=True,
)

Design ID               : df4
Design name             : prep_genus_count
Description             : Preprocessed HIVRC, genus-level, raw count
Aggregation             : genus
Normalize               : False
Cleanset Filtering      : True
Subset studies          : None
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table           : (936, 668)
meta_data               : (936, 11)
n_batches               : 14


# Trial investigation

In [3]:
import pandas as pd
import glob

# df4 CSV 파일 찾기
df4_p1_res = pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df4/phase1/optuna_trials_2026-05-15.csv")
print(f"전체 trial 수: {len(df4_p1_res)}")

# 필터링
df_clean = df4_p1_res[
    (df4_p1_res['permanova'] > 0.01) &
    (df4_p1_res['permanova'] < 0.1) &
    (df4_p1_res['noise_ratio'] < 0.65) &
    (df4_p1_res['n_clusters'] >= 10)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'], ascending=[True, True, False])

print(f"의미있는 trial 수: {len(df_clean)}")
print(df_clean[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']].head(20))

전체 trial 수: 1344
의미있는 trial 수: 0
Empty DataFrame
Columns: [trial_number, permanova, n_clusters, noise_ratio, cutoff]
Index: []


In [4]:
import optuna

study = optuna.load_study(
    study_name="latentgee_optuna_df4",
    storage="sqlite:////DATA/WGS_study/YSL/projects/latentgee/examples/latentgee_optuna.db"
)

# 상태별 trial 수 확인
from collections import Counter
state_counts = Counter(str(t.state) for t in study.trials)
for state, count in state_counts.items():
    print(f"{state}: {count}")

TrialState.PRUNED: 108
TrialState.COMPLETE: 1892


In [6]:
# n_clusters 기준 완화
for min_k in [10, 8, 5, 3]:
    filtered = df4_p1_res[
        (df4_p1_res['permanova'] > 0.01) &
        (df4_p1_res['permanova'] < 0.1) &
        (df4_p1_res['noise_ratio'] < 0.65) &
        (df4_p1_res['n_clusters'] >= min_k)
    ]
    print(f"n_clusters >={min_k}: {len(filtered)}개")


# 조건 완화한 best trial 확인
df_clean2 = df4_p1_res[
        (df4_p1_res['permanova'] > 0.01) &
        (df4_p1_res['permanova'] < 0.1) &
        (df4_p1_res['noise_ratio'] < 0.65)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'], ascending=[True, True, False]).head(20)

print(f"\n완화된 필터 결과: {len(df_clean2)}개")
print(df_clean2[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']])

n_clusters >=10: 0개
n_clusters >=8: 0개
n_clusters >=5: 0개
n_clusters >=3: 7개

완화된 필터 결과: 20개
     trial_number  permanova  n_clusters  noise_ratio  cutoff
33            104   0.087942           2     0.053419   0.005
189           395   0.044840           2     0.349359   0.050
114           232   0.085338           3     0.469017   0.050
119           245   0.051690           2     0.482906   0.070
162           342   0.041881           2     0.504274   0.100
170           350   0.053865           2     0.508547   0.100
78            193   0.056603           3     0.513889   0.005
86            206   0.050899           3     0.517094   0.070
110           238   0.052289           2     0.518162   0.070
59            153   0.044526           3     0.529915   0.100
39            112   0.056815           4     0.529915   0.005
77            194   0.046370           2     0.547009   0.070
87            207   0.061476           2     0.564103   0.070
176           367   0.073495           

# Best trial selection (trial 104)

In [7]:
print(df4_p1_res[df4_p1_res['trial_number'] == 104].T)

                              33
cutoff                     0.005
trial_number                 104
base_dim                     768
n_layers                       3
latent_dim                    49
activation                  relu
strategy                constant
dropout_rate                 0.2
epochs                       100
learning_rate           0.000026
loss                    0.676937
permanova               0.087942
n_clusters                     2
noise_ratio             0.053419
min_cluster_size               3
min_samples_token            NaN
batch_size                   128
init               xavier_normal
beta_kl                 0.010069
weight_decay                 0.0
grad_clip_norm          0.704907
csm                          eom
kl_warmup_ratio         0.421161
norm                   layernorm
scheduler                 cosine


# Run phase 2

In [8]:
from prototype_updated_phase2 import main
main(
    experiment_name="df4",
    phase=2,
    best_trial_number=104,
    trial_res_file_phase2="/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df4/phase1/optuna_trials_2026-05-15.csv"
)

2026-05-18 13:44:21 | 로그 저장 경로: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df4/phase2/latentgee_prototype_cutoff_df4_pid2032756_2026-05-18.log
2026-05-18 13:44:21 | LatentGEE 시작 | experiment=df4 | phase=2
2026-05-18 13:44:21 | config snapshot saved: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df4/phase2/config_used.yaml
2026-05-18 13:44:21 | python == 3.10.20
2026-05-18 13:44:21 | torch == 2.2.2
2026-05-18 13:44:21 | numpy == 1.26.4
2026-05-18 13:44:21 | scikit-learn == 1.7.2
2026-05-18 13:44:21 | optuna == 3.6.1
2026-05-18 13:44:21 | hdbscan == 0.8.41
Design ID               : df4
Design name             : prep_genus_count
Description             : Preprocessed HIVRC, genus-level, raw count
Aggregation             : genus
Normalize               : False
Cleanset Filtering      : True
Subset studies          : None
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table        

In [9]:
# from functions import zero_filter
def zero_filter(df, meta, cutoff):
    batch="Study"
    prevalence = (df > 0).sum(axis=0) / df.shape[0]
    df = df.loc[:, prevalence > best_cutoff].copy()

    row_sums = df.sum(axis=1)
    keep_sample = row_sums > 0
    df = df.loc[keep_sample].copy()
    meta = meta.loc[keep_sample.values].reset_index(drop=True)
    
    assert len(df) == len(meta)
    assert all(df.index.astype(str) == meta["SeqID"].astype(str))
    
    return df, meta, meta[batch]

best_cutoff = 0.005
X_df4_filtered, meta_df4_filtered, batch_df4_filtered = zero_filter(X_df4, meta_df4, best_cutoff)

    

In [12]:
X_df4_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df4_filtered_cutoff0.005.csv", index=True)
meta_df4_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/meta_df4_filtered_cutoff0.005.csv", index=True)
batch_df4_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/batch_df4_filtered_cutoff0.005.csv", index=True)

In [3]:
import numpy as np
import pandas as pd
# X_corrected 로드
X_corrected_df4 = pd.read_csv(
    "/DATA/WGS_study/YSL/projects/latentgee/examples/results/df4/phase2/df4_X_corrected_trial104_cutoff0.005_2026-05-18.csv",
    index_col=0
)
# inf, NaN 처리
X_corrected_df4_clean = X_corrected_df4.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
row_sums = X_corrected_df4_clean.sum(axis=1).replace(0, 1)
X_corrected_df4_clean = X_corrected_df4_clean.div(row_sums, axis=0)

print(f"shape: {X_corrected_df4_clean.shape}")
print(f"NaN 수: {X_corrected_df4_clean.isna().sum().sum()}")
print(f"inf 수: {np.isinf(X_corrected_df4_clean.values).sum()}")

X_corrected_df4_clean.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/results/df4/phase2/X_corrected_df4_clean_latentgee.csv", index=True)

shape: (936, 353)
NaN 수: 0
inf 수: 0


In [16]:
from functions import evaluate_batch_correction
df4_result = evaluate_batch_correction(
    X_before     = X_df4_filtered,
    X_after      = X_corrected_df4_clean,
    batch_labels = batch_df4_filtered,
    bio_labels   = meta_df4_filtered['hivstatus'],
    renormalize  = False,
    label        = "df4 — LatentGEE"
)



  df4 — LatentGEE
                        Before   After  Change
Metric                                        
PERMANOVA R² (batch) ↓  0.2649  0.0241 -0.2408
PERMANOVA R² (bio) ↑    0.0039  0.0011 -0.0027
kBET acceptance rate ↑  0.0085  0.8098  0.8013
ASW (batch) → 0        -0.1329 -0.0676  0.0654
ASW (bio) ↑            -0.0261 -0.0001  0.0260

